#Medlamingo — Backbone Choice (LLaMA‑7B vs MPT‑7B) + ZERO‑SHOT over JSONL



## 0) Install & Imports

In [ ]:

import torch, torchvision
print("torch     :", torch.__version__)
print("torchvision:", torchvision.__version__)



In [ ]:
# %pip install -U open-flamingo[all] transformers accelerate pandas pillow --index-url https://download.pytorch.org/whl/cu121
import os, json
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
from open_flamingo import create_model_and_transforms
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1) Choose Backbone

In [ ]:
import torch
from open_flamingo import create_model_and_transforms

device = "cuda" if torch.cuda.is_available() else "cpu"

model, image_processor, tokenizer = create_model_and_transforms(
    clip_vision_encoder_path="ViT-L-14",
    clip_vision_encoder_pretrained="openai",  # already cached
    lang_encoder_path="models/llama-7b",
    tokenizer_path="models/llama-7b",
    cross_attn_every_n_layers=1,
    use_local_files=True,          # <-- ensure it doesn’t try to fetch online
    cache_dir="models/llama-7b",   # <-- optional: keep everything together
)

ckpt_path = "checkpoints/openflamingo-9b/checkpoint.pt"
state = torch.load(ckpt_path, map_location="cpu")
_ = model.load_state_dict(state, strict=False)
model = model.to(device).eval()
tokenizer.padding_side = "left"
print("✅ OpenFlamingo (LLaMA) loaded fully offline")


## 2) Provide Your Dataset (JSONL or inline)

In [ ]:
JSONL_PATH = Path('./train_flat.jsonl')
inline_examples = [
  {"image_path": "../RadSpineXR/Train/Unannotated_images/vindr_train_019.png", "question": "Is there any abnormality present in this image?", "answer": "No abnormality detected."},
  {"image_path": "../RadSpineXR/Train/Unannotated_images/vindr_train_027.png", "question": "What is the nature of the abnormality observed in the X-ray image?", "answer": "The X-ray image shows a fracture of the C6 vertebra, characterized by a clear break in the bone structure."},
  {"image_path": "../RadSpineXR/Train/Unannotated_images/vindr_train_027.png", "question": "What is the severity of the fracture observed in the X-ray image?", "answer": "The fracture appears to be a significant displacement of the C6 vertebra, indicating a high severity."},
  {"image_path": "../RadSpineXR/Train/Unannotated_images/vindr_train_027.png", "question": "Where is the fracture located in the X-ray image?", "answer": "The fracture is located at the C6 vertebra, which is in the cervical spine region."},
  {"image_path": "../RadSpineXR/Train/Unannotated_images/vindr_train_027.png", "question": "What is the most likely diagnosis based on the X-ray image?", "answer": "The most likely diagnosis is a cervical spine fracture, specifically at the C6 vertebra."},
]
with open(JSONL_PATH, 'w', encoding='utf-8') as f:
    for rec in inline_examples:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print('Prepared:', JSONL_PATH)

## 3) ZERO‑SHOT Batch Inference

In [ ]:
# -------------------- MedFlamingo Zero-shot on test set --------------------
# pip installs (uncomment if needed)
# %pip install -U open-flamingo[all] transformers accelerate pandas pillow --index-url https://download.pytorch.org/whl/cu121

import os, json, traceback
from pathlib import Path

import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm

from open_flamingo import create_model_and_transforms
from transformers import AutoTokenizer, StoppingCriteria, StoppingCriteriaList

# -------------------- CONFIG --------------------
JSONL_PATH = Path("../MedGemma/test_flat.jsonl")     # <-- full test set
OUT_CSV    = Path("./ZeroShot_test.csv")

# Practical high cap for MedFlamingo on LLaMA; raise/lower if you like.
# (True "max" is bounded by model context, but 256 is a solid upper bound for answers
# without exploding VRAM or latency on most checkpoints.)
MAX_NEW_TOKENS = 256

TEMPERATURE = 0.0
TOP_P = 1.0
REPETITION_PENALTY = 1.0

# VRAM options
FORCE_FP16 = True
FORCE_BF16 = False   # set True on GPUs that prefer bf16 (A100/H100/RTX 4xxx)

# Local paths for your setup
LANG_ENCODER_PATH = "models/llama-7b"
TOKENIZER_PATH    = "models/llama-7b"
CHECKPOINT_PATH   = "checkpoints/openflamingo-9b/checkpoint.pt"  # adjust if needed

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -------------------- BOOTSTRAP MODEL --------------------
model, image_processor, tokenizer = create_model_and_transforms(
    clip_vision_encoder_path="ViT-L-14",
    clip_vision_encoder_pretrained="openai",
    lang_encoder_path=LANG_ENCODER_PATH,
    tokenizer_path=TOKENIZER_PATH,
    cross_attn_every_n_layers=1,
    use_local_files=True,
    cache_dir=LANG_ENCODER_PATH,
)
if isinstance(tokenizer, AutoTokenizer):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

if CHECKPOINT_PATH and Path(CHECKPOINT_PATH).exists():
    print(f">> Loading checkpoint from: {CHECKPOINT_PATH}")
    state = torch.load(CHECKPOINT_PATH, map_location="cpu")
    model.load_state_dict(state, strict=False)

model = model.to(device).eval()
torch.set_grad_enabled(False)

if FORCE_FP16 and not FORCE_BF16:
    print(">> Casting model to fp16")
    model = model.half()
elif FORCE_BF16:
    print(">> Casting model to bf16")
    model = model.to(dtype=torch.bfloat16)

# -------------------- HELPERS --------------------
def _stack_like(px_list):
    """Stack a list/tuple of image-like items into (B,C,H,W) tensor."""
    if len(px_list) > 0 and isinstance(px_list[0], dict) and "pixel_values" in px_list[0]:
        px_list = [d["pixel_values"] for d in px_list]
    t_list = []
    for x in px_list:
        t_list.append(x if isinstance(x, torch.Tensor) else torch.as_tensor(x))
    t = torch.stack(t_list)
    # (B,H,W,C) -> (B,C,H,W)
    if t.ndim == 4 and t.shape[-1] in (1, 3):
        t = t.permute(0, 3, 1, 2).contiguous()
    return t

def to_bchw1_debug(im, image_processor, device):
    """Convert PIL.Image -> (1,C,H,W) and print a few shapes (first sample prints are handy)."""
    out = image_processor(im)
    # dict with pixel_values
    if isinstance(out, dict) and "pixel_values" in out:
        px = out["pixel_values"]
        if isinstance(px, torch.Tensor):
            t = px if px.ndim == 4 else px.unsqueeze(0)
        elif isinstance(px, (list, tuple)):
            t = _stack_like(px)
        else:
            raise TypeError(f"Unsupported pixel_values type: {type(px)}")
    # tensor
    elif isinstance(out, torch.Tensor):
        t = out if out.ndim == 4 else out.unsqueeze(0)
    # list/tuple
    elif isinstance(out, (list, tuple)):
        t = _stack_like(out)
    else:
        # Fallback CLIP-like transform
        import torchvision.transforms as T
        tf = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                        std=[0.26862954, 0.26130258, 0.27577711]),
        ])
        t = tf(im).unsqueeze(0)

    if t.ndim == 4 and t.shape[-1] in (1, 3):  # (B,H,W,C) -> (B,C,H,W)
        t = t.permute(0,3,1,2).contiguous()
    if t.ndim == 3:
        t = t.unsqueeze(0)
    return t.to(device)

def make_vision_x6d(img_bchw, model, device, T_img=1, F=1):
    """Cast to vision encoder dtype and convert to (B,T_img,F,C,H,W) for MedFlamingo."""
    assert img_bchw.ndim == 4, f"expected BCHW, got {img_bchw.ndim}D"
    try:
        vision_dtype = next(model.vision_encoder.parameters()).dtype
    except Exception:
        vision_dtype = img_bchw.dtype
    img_bchw = img_bchw.to(device=device, dtype=vision_dtype)
    v = img_bchw.unsqueeze(1).unsqueeze(2)  # (B,1,1,C,H,W)
    if T_img != 1 or F != 1:
        v = v.repeat(1, T_img, F, 1, 1, 1)
    return v

# ----- stop criteria + cleaner to prevent truncation mess -----
STOP_STRINGS = ["<|endofchunk|>", "\nQ:", "\nQuestion:", "\n\n"]
_stop_token_ids = [tokenizer(s, add_special_tokens=False).input_ids for s in STOP_STRINGS]
_stop_token_ids = [ids if isinstance(ids, list) else [ids] for ids in _stop_token_ids]

class StopOnTokens(StoppingCriteria):
    def __init__(self, stop_seqs):
        super().__init__()
        self.stop_seqs = stop_seqs  # list[list[int]]

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs):
        for stop in self.stop_seqs:
            L = len(stop)
            if L == 0:
                continue
            if input_ids.shape[1] >= L and input_ids[0, -L:].tolist() == stop:
                return True
        return False

stops = StoppingCriteriaList([StopOnTokens(_stop_token_ids)])

def clean_answer(text: str) -> str:
    cut = len(text)
    for s in STOP_STRINGS:
        i = text.find(s)
        if i != -1:
            cut = min(cut, i)
    return text[:cut].strip().strip(" ,.;")

# -------------------- PREDICT ONE --------------------
def predict_one(image_path, question, model, tokenizer, image_processor, device, max_new_tokens=256):
    im = Image.open(image_path).convert("RGB")
    img_bchw = to_bchw1_debug(im, image_processor, device)     # (1,C,H,W)
    vision_x = make_vision_x6d(img_bchw, model, device)        # (1,1,1,C,H,W)

    prompt = f"<image>Q: {question} A: <|endofchunk|>"
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
    enc = {k: v.to(device) for k, v in enc.items()}

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        num_beams=1,
        do_sample=False,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY,
        use_cache=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        stopping_criteria=stops,
    )

    with torch.no_grad():
        try:
            amp_dtype = next(model.parameters()).dtype
        except StopIteration:
            amp_dtype = torch.float16
        with torch.cuda.amp.autocast(enabled=(device == "cuda"), dtype=amp_dtype):
            out = model.generate(
                vision_x=vision_x,
                lang_x=enc["input_ids"],
                attention_mask=enc.get("attention_mask", None),
                **gen_kwargs,
            )

    full = tokenizer.decode(out[0], skip_special_tokens=False)
    ans = full.split("<|endofchunk|>")[-1]
    ans = clean_answer(ans)

    # Clean up
    del vision_x, img_bchw, enc, out
    if device == "cuda":
        torch.cuda.empty_cache()
    return ans

# -------------------- RUN ON TEST SET --------------------
rows = []
with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(tqdm(f, desc="Zero-shot (test)")):
        rec = json.loads(line)
        image_path = rec["image_path"]
        question = rec["question"]
        gt = rec.get("answer", "")

        try:
            pred = predict_one(
                image_path=image_path,
                question=question,
                model=model,
                tokenizer=tokenizer,
                image_processor=image_processor,
                device=device,
                max_new_tokens=MAX_NEW_TOKENS,
            )
        except Exception as e:
            pred = f"ERROR: {e}"

        rows.append({
            "image_path": image_path,
            "question": question,
            "ground_truth": gt,
            "prediction": pred,
        })

        if device == "cuda":
            torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print("✅ Saved to", OUT_CSV.resolve())
# ---------------------------------------------------------------------------


### Finetuning Script


In [1]:
# ==============================================================
# 🔥 MedFlamingo Fine-tuning (Offline) with TRL SFTTrainer (ONE-SHOT FIX)
# ==============================================================

# %pip install -U open-flamingo[all] transformers accelerate peft trl pandas pillow tqdm --index-url https://download.pytorch.org/whl/cu121

import os, json
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from PIL import Image

from transformers import TrainingArguments, GenerationConfig
from trl import SFTTrainer
from open_flamingo import create_model_and_transforms
# 🔧 Force usage of the second A6000 (GPU 1)
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # logical cuda:0 -> physical GPU 1

# ---------------- Config ----------------
CFG = {
    "TRAIN_JSONL": "../MedGemma/train_flat.jsonl",
    "VAL_JSONL":   "../MedGemma/val_flat.jsonl",
    "IMAGE_BASE":  ".",

    "CLIP_VISION_ENCODER_PATH": "ViT-L-14",
    "CLIP_VISION_PRETRAINED":   "openai",
    "LANG_ENCODER_PATH":        "models/llama-7b",
    "TOKENIZER_PATH":           "models/llama-7b",
    "CHECKPOINT_PATH":          "checkpoints/openflamingo-9b/checkpoint.pt",

    "SEED": 42,
    "EPOCHS": 3,
    "PER_DEVICE_TRAIN_BATCH_SIZE": 1,
    "PER_DEVICE_EVAL_BATCH_SIZE":  1,
    "GRAD_ACCUM_STEPS": 8,
    "LR": 2e-5,
    "WEIGHT_DECAY": 0.01,
    "WARMUP_RATIO": 0.05,
    "MAX_TEXT_LEN": 384,
    "CLIP_GRAD_NORM": 1.0,

    "USE_BF16": False,
    "USE_FP16": False,
    "TF32": True,

    "USE_LORA": True,
    "LORA_R": 8,
    "LORA_ALPHA": 16,
    "LORA_DROPOUT": 0.05,

    "OUT_DIR": "./medflamingo_sft_out",
    "SAVE_STEPS": 1000,
    "EVAL_STEPS": 1000,
    "LOGGING_STEPS": 50,
    "SAVE_TOTAL_LIMIT": 3,
}

# ---------------- Repro / precision ----------------
if CFG["TF32"]:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

def set_seed(seed=42):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(CFG["SEED"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------------- Load Flamingo + tokenizer ----------------
assert Path(CFG["LANG_ENCODER_PATH"]).exists()
assert Path(CFG["TOKENIZER_PATH"]).exists()
assert Path(CFG["CHECKPOINT_PATH"]).exists()

model, image_processor, tokenizer = create_model_and_transforms(
    clip_vision_encoder_path=CFG["CLIP_VISION_ENCODER_PATH"],
    clip_vision_encoder_pretrained=CFG["CLIP_VISION_PRETRAINED"],
    lang_encoder_path=CFG["LANG_ENCODER_PATH"],
    tokenizer_path=CFG["TOKENIZER_PATH"],
    cross_attn_every_n_layers=1,
    use_local_files=True,
    cache_dir=CFG["LANG_ENCODER_PATH"],
)

tokenizer.padding_side = "left"
if tokenizer.pad_token is None and hasattr(tokenizer, "eos_token"):
    tokenizer.pad_token = tokenizer.eos_token

print(f">> Loading OpenFlamingo checkpoint: {CFG['CHECKPOINT_PATH']}")
state = torch.load(CFG["CHECKPOINT_PATH"], map_location="cpu", weights_only=True)
missing, unexpected = model.load_state_dict(state, strict=False)
if missing:    print("Missing keys:", len(missing))
if unexpected: print("Unexpected keys:", len(unexpected))

# ---------------- Optional LoRA on language model ----------------
if CFG["USE_LORA"]:
    from peft import LoraConfig, get_peft_model, TaskType
    lang_model = getattr(model, "lang_encoder", None)
    if lang_model is None:
        raise RuntimeError("Could not find model.lang_encoder for LoRA")
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=CFG["LORA_R"],
        lora_alpha=CFG["LORA_ALPHA"],
        lora_dropout=CFG["LORA_DROPOUT"],
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        bias="none",
    )
    model.lang_encoder = get_peft_model(lang_model, lora_cfg)
    model.lang_encoder.print_trainable_parameters()
else:
    for p in model.parameters(): p.requires_grad = True

# ---------------- Precision cast (after LoRA) ----------------
if CFG["USE_BF16"]:
    model = model.to(device=device, dtype=torch.bfloat16)
elif CFG["USE_FP16"]:
    model = model.to(device)
    model = model.to(dtype=torch.float16)
else:
    model = model.to(device)

model.train()

# ---------------- HF-compat wrapper (adds .config & generation_config) ----------------
from types import SimpleNamespace

class FlamingoHFWrapper(nn.Module):
    def __init__(self, inner, tokenizer):
        super().__init__()
        self.inner = inner
        torch_dtype = next(inner.parameters()).dtype

        # Minimal config fields HuggingFace Trainer/TRL poke at
        self.config = SimpleNamespace(
            _attn_implementation="eager",
            torch_dtype=torch_dtype,
            use_cache=False,
            is_encoder_decoder=False,
            pad_token_id=getattr(tokenizer, "pad_token_id", None),
            bos_token_id=getattr(tokenizer, "bos_token_id", None),
            eos_token_id=getattr(tokenizer, "eos_token_id", None),
            vocab_size=getattr(tokenizer, "vocab_size", None),
        )
        # Generation config that aligns with tokenizer ids
        self.generation_config = GenerationConfig(
            pad_token_id=self.config.pad_token_id,
            bos_token_id=self.config.bos_token_id,
            eos_token_id=self.config.eos_token_id,
        )

    def forward(self, **kwargs):
        # Delegate to Flamingo; kwargs contain vision_x, lang_x, attention_mask, labels
        return self.inner(**kwargs)

    # HF Trainer may call this during special-token alignment; make it a no-op
    def resize_token_embeddings(self, *args, **kwargs):
        return None

    @property
    def device(self):
        return next(self.inner.parameters()).device

hf_model = FlamingoHFWrapper(model, tokenizer)

# ---------------- Vision helpers ----------------
def _stack_like(px_list):
    if len(px_list) > 0 and isinstance(px_list[0], dict) and "pixel_values" in px_list[0]:
        px_list = [d["pixel_values"] for d in px_list]
    t_list = [(x if isinstance(x, torch.Tensor) else torch.as_tensor(x)) for x in px_list]
    t = torch.stack(t_list)
    if t.ndim == 4 and t.shape[-1] in (1, 3): t = t.permute(0, 3, 1, 2).contiguous()
    return t

def to_bchw1(im: Image.Image, image_processor):
    out = image_processor(im)
    if isinstance(out, dict) and "pixel_values" in out:
        px = out["pixel_values"]
        if isinstance(px, torch.Tensor): t = px if px.ndim == 4 else px.unsqueeze(0)
        elif isinstance(px, (list, tuple)): t = _stack_like(px)
        else: raise TypeError(f"Unsupported pixel_values type: {type(px)}")
    elif isinstance(out, torch.Tensor):
        t = out if out.ndim == 4 else out.unsqueeze(0)
    elif isinstance(out, (list, tuple)):
        t = _stack_like(out)
    else:
        import torchvision.transforms as T
        tf = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                        std=[0.26862954, 0.26130258, 0.27577711]),
        ])
        t = tf(im).unsqueeze(0)
    if t.ndim == 4 and t.shape[-1] in (1, 3): t = t.permute(0,3,1,2).contiguous()
    if t.ndim == 3: t = t.unsqueeze(0)
    vis_dtype = next(model.vision_encoder.parameters()).dtype
    return t.to(dtype=vis_dtype)

def make_vision_x6d(img_bchw, T_img=1, F=1):
    v = img_bchw.unsqueeze(1).unsqueeze(2)
    if T_img != 1 or F != 1: v = v.repeat(1, T_img, F, 1, 1, 1)
    return v

# ---------------- Dataset & Collator ----------------
class VQADataset(Dataset):
    def __init__(self, jsonl_path: str | Path, image_base=".", verify_exist=False, max_len=CFG["MAX_TEXT_LEN"]):
        self.items = []
        self.max_len = max_len
        self.image_base = Path(image_base)
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                ex = json.loads(line)
                img = ex["image_path"]; q = ex["question"]
                a = ex.get("answer", ex.get("ground_truth", ""))
                p = Path(img); p = p if p.is_absolute() else (self.image_base / p)
                if verify_exist and not p.exists(): continue
                self.items.append({"image_path": str(p), "question": q, "answer": a, "dummy_text": ""})
    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

@dataclass
class FlamingoCollator:
    tokenizer: Any
    image_processor: Any
    max_len: int
    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        prompts, answers, imgs = [], [], []
        for ex in batch:
            q = ex["question"]; a = ex["answer"] if ex["answer"] is not None else ""
            prompts.append(f"<image>Q: {q} A: "); answers.append(a)
            im = Image.open(ex["image_path"]).convert("RGB")
            imgs.append(to_bchw1(im, self.image_processor))
        tok_full = self.tokenizer([p + ans for p, ans in zip(prompts, answers)],
                                  padding=True, truncation=True, max_length=self.max_len, return_tensors="pt")
        input_ids, attention_mask = tok_full["input_ids"], tok_full["attention_mask"]
        labels = input_ids.clone()
        for i, p in enumerate(prompts):
            p_ids = self.tokenizer(p, add_special_tokens=False).input_ids
            seq = input_ids[i].tolist()
            pad_id = self.tokenizer.pad_token_id
            first_nonpad = 0
            if pad_id is not None:
                while first_nonpad < len(seq) and seq[first_nonpad] == pad_id: first_nonpad += 1
            prompt_end = first_nonpad + min(len(p_ids), len(seq) - first_nonpad)
            labels[i, first_nonpad:prompt_end] = -100
            labels[i, attention_mask[i] == 0] = -100
        img_batch = torch.cat(imgs, dim=0)
        vision_x  = make_vision_x6d(img_batch, 1, 1)
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels, "vision_x": vision_x}

# ---------------- Build datasets ----------------
train_ds = VQADataset(CFG["TRAIN_JSONL"], image_base=CFG["IMAGE_BASE"])
val_ds: Optional[Dataset] = None
if CFG["VAL_JSONL"] and Path(CFG["VAL_JSONL"]).exists():
    val_ds = VQADataset(CFG["VAL_JSONL"], image_base=CFG["IMAGE_BASE"])
collator = FlamingoCollator(tokenizer, image_processor, CFG["MAX_TEXT_LEN"])
print(f"Train samples: {len(train_ds)}")
if val_ds: print(f"Val samples:   {len(val_ds)}")

# ---------------- Vision-aware SFTTrainer ----------------
from trl.trainer.sft_trainer import SFTTrainer as _SFTBase
class VisionSFTTrainer(_SFTBase):
    # Match TRL's signature and bypass its dataset formatting logic
    def _prepare_dataset(
        self,
        dataset,
        processing_class=None,
        args=None,
        packing=None,
        formatting_func=None,
        split=None,
    ):
        return dataset

    # 🔧 FIXED SIGNATURE: accept num_items_in_batch and ignore it
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs: bool = False,
        num_items_in_batch: int | None = None,  # <--- added arg
    ):
        out = model(
            lang_x=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            vision_x=inputs["vision_x"],
            labels=inputs.get("labels"),
        )

        loss = getattr(out, "loss", None)
        if loss is None:
            logits = out.logits
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = inputs["labels"][:, 1:].contiguous()
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )

        return (loss, out) if return_outputs else loss

# ---------------- TrainingArguments ----------------
eval_strat = "steps" if val_ds is not None else "no"
args = TrainingArguments(
    output_dir=CFG["OUT_DIR"],
    num_train_epochs=CFG["EPOCHS"],
    per_device_train_batch_size=CFG["PER_DEVICE_TRAIN_BATCH_SIZE"],
    per_device_eval_batch_size=CFG["PER_DEVICE_EVAL_BATCH_SIZE"],
    gradient_accumulation_steps=CFG["GRAD_ACCUM_STEPS"],
    learning_rate=CFG["LR"],
    weight_decay=CFG["WEIGHT_DECAY"],
    warmup_ratio=CFG["WARMUP_RATIO"],
    lr_scheduler_type="cosine",
    max_grad_norm=CFG["CLIP_GRAD_NORM"],
    fp16=CFG["USE_FP16"],
    bf16=CFG["USE_BF16"],
    logging_steps=CFG["LOGGING_STEPS"],
    eval_strategy=("steps" if val_ds is not None else "no"),
    save_strategy="steps",
    eval_steps=CFG["EVAL_STEPS"] if val_ds is not None else None,
    save_steps=CFG["SAVE_STEPS"],
    save_total_limit=CFG["SAVE_TOTAL_LIMIT"],
    load_best_model_at_end=bool(val_ds),
    metric_for_best_model="loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
)
# ---------------- Initialize trainer & train ----------------
trainer = VisionSFTTrainer(
    model=hf_model,                 # use the HF wrapper
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    processing_class=tokenizer,     # avoid AutoProcessor(model.config)
)

train_result = trainer.train()
print("Training done.")
metrics = train_result.metrics
print("Final train metrics:", metrics)

if val_ds is not None:
    eval_metrics = trainer.evaluate()
    print("Final eval metrics:", eval_metrics)

# ---------------- Save ----------------
trainer.save_model(CFG["OUT_DIR"])
tokenizer.save_pretrained(CFG["OUT_DIR"])

if CFG["USE_LORA"]:
    try:
        model.lang_encoder.save_pretrained(Path(CFG["OUT_DIR"]) / "lora_lang_encoder")
        print("💾 Saved LoRA adapters to", Path(CFG["OUT_DIR"]) / "lora_lang_encoder")
    except Exception as e:
        print("LoRA save warning:", e)

print("✅ All done. Outputs at:", Path(CFG["OUT_DIR"]).resolve())


Device: cuda


/home/deepalim/work/MedFlamingo/Medflaming_env/lib/python3.10/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior wil

Flamingo model initialized with 4657367104 trainable parameters
>> Loading OpenFlamingo checkpoint: checkpoints/openflamingo-9b/checkpoint.pt
Missing keys: 1402
Unexpected keys: 34
trainable params: 19,988,480 || all params: 11,221,692,480 || trainable%: 0.1781
Train samples: 2228
Val samples:   412


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 47.40 GiB of which 22.62 MiB is free. Including non-PyTorch memory, this process has 47.36 GiB memory in use. Of the allocated memory 46.82 GiB is allocated by PyTorch, and 219.14 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Evalaution

In [6]:
# %% Install (run once in your env)
# %pip install pandas nltk rouge-score bert-score 

import pandas as pd
from pathlib import Path

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score

# ---------- Config ----------
CSV_PATH = Path("./ZeroShot_test.csv")

# ---------- Load data ----------
df = pd.read_csv(CSV_PATH)

# If your columns are named differently, adjust here:
REF_COL = "ground_truth"
PRED_COL = "prediction"

# Optional: filter out rows with obvious errors
df = df.dropna(subset=[REF_COL, PRED_COL])
df = df[~df[PRED_COL].astype(str).str.startswith("[ERROR]")].reset_index(drop=True)

refs_raw = df[REF_COL].astype(str).tolist()
preds_raw = df[PRED_COL].astype(str).tolist()

print(f"Using {len(refs_raw)} examples for evaluation")

# ---------- Accuracy (exact match, normalized) ----------
def normalize_text(s: str) -> str:
    return " ".join(s.strip().lower().split())

ref_norm = [normalize_text(x) for x in refs_raw]
pred_norm = [normalize_text(x) for x in preds_raw]

correct = sum(r == p for r, p in zip(ref_norm, pred_norm))
accuracy = correct / len(ref_norm) if ref_norm else 0.0
print(f"Accuracy (exact match): {accuracy:.4f}  ({correct}/{len(ref_norm)})")

# ---------- BLEU (corpus BLEU) ----------
# NLTK expects: references: list of list of ref-token-lists, hypotheses: list of token-lists
refs_tok = [[r.split()] for r in ref_norm]
preds_tok = [p.split() for p in pred_norm]

smooth = SmoothingFunction().method1
bleu_score = corpus_bleu(refs_tok, preds_tok, smoothing_function=smooth)
print(f"BLEU (corpus): {bleu_score:.4f}")

# ---------- ROUGE-L ----------
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

rouge_l_f = []
for r, p in zip(refs_raw, preds_raw):
    score = scorer.score(r, p)["rougeL"]
    rouge_l_f.append(score.fmeasure)

rouge_l_avg = sum(rouge_l_f) / len(rouge_l_f) if rouge_l_f else 0.0
print(f"ROUGE-L (F1, avg): {rouge_l_avg:.4f}")

# ---------- BERTScore ----------
# This can be slow on CPU; by default it will use GPU if available.
# You can change 'model_type' if you want a specific encoder.
print("Computing BERTScore (this may take a bit)...")
P, R, F1 = bertscore_score(preds_raw, refs_raw, rescale_with_baseline=False, model_type="bert-base-uncased")

bert_p = float(P.mean())
bert_r = float(R.mean())
bert_f1 = float(F1.mean())

print(f"BERTScore Precision: {bert_p:.4f}")
print(f"BERTScore Recall   : {bert_r:.4f}")
print(f"BERTScore F1       : {bert_f1:.4f}")


Using 535 examples for evaluation
Accuracy (exact match): 0.0000  (0/535)
BLEU (corpus): 0.0079
ROUGE-L (F1, avg): 0.0750
Computing BERTScore (this may take a bit)...
BERTScore Precision: 0.3858
BERTScore Recall   : 0.4528
BERTScore F1       : 0.4129


In [1]:
!pip install rouge-score -qqq


In [2]:
# -------------------- MedFlamingo Zero-shot on test set --------------------
# pip installs (uncomment if needed)
# %pip install -U open-flamingo[all] transformers accelerate pandas pillow --index-url https://download.pytorch.org/whl/cu121

import os, json, traceback
from pathlib import Path

import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm

from open_flamingo import create_model_and_transforms
from transformers import AutoTokenizer, StoppingCriteria, StoppingCriteriaList

# -------------------- CONFIG --------------------
JSONL_PATH = Path("../MedGemma/test_flat.jsonl")     # <-- full test set
OUT_CSV    = Path("./test.csv")

# Max new tokens for answer generation (output length)
MAX_NEW_TOKENS   = 128     # 128 is usually enough for MedVQA; you can push to 256 if needed

# Max tokens for the text prompt fed into LLaMA (input length)
MAX_INPUT_TOKENS = 512

TEMPERATURE         = 0.0
TOP_P               = 1.0
REPETITION_PENALTY = 1.0

# VRAM options
FORCE_FP16 = True
FORCE_BF16 = False   # set True on GPUs that prefer bf16 (A100/H100/RTX 4xxx)

# Local paths for your setup
LANG_ENCODER_PATH = "models/llama-7b"
TOKENIZER_PATH    = "models/llama-7b"
CHECKPOINT_PATH   = "checkpoints/openflamingo-9b/checkpoint.pt"  # adjust if needed

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -------------------- BOOTSTRAP MODEL --------------------
model, image_processor, tokenizer = create_model_and_transforms(
    clip_vision_encoder_path="ViT-L-14",
    clip_vision_encoder_pretrained="openai",
    lang_encoder_path=LANG_ENCODER_PATH,
    tokenizer_path=TOKENIZER_PATH,
    cross_attn_every_n_layers=1,
    use_local_files=True,
    cache_dir=LANG_ENCODER_PATH,
)

if isinstance(tokenizer, AutoTokenizer):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

if CHECKPOINT_PATH and Path(CHECKPOINT_PATH).exists():
    print(f">> Loading checkpoint from: {CHECKPOINT_PATH}")
    state = torch.load(CHECKPOINT_PATH, map_location="cpu")
    model.load_state_dict(state, strict=False)

model = model.to(device).eval()
torch.set_grad_enabled(False)

if FORCE_FP16 and not FORCE_BF16:
    print(">> Casting model to fp16")
    model = model.half()
elif FORCE_BF16:
    print(">> Casting model to bf16")
    model = model.to(dtype=torch.bfloat16)

print("Model dtype:", next(model.parameters()).dtype)
print("Tokenizer pad_token:", tokenizer.pad_token, "| eos_token:", tokenizer.eos_token)

# -------------------- HELPERS --------------------
def _stack_like(px_list):
    """Stack a list/tuple of image-like items into (B,C,H,W) tensor."""
    if len(px_list) > 0 and isinstance(px_list[0], dict) and "pixel_values" in px_list[0]:
        px_list = [d["pixel_values"] for d in px_list]
    t_list = []
    for x in px_list:
        t_list.append(x if isinstance(x, torch.Tensor) else torch.as_tensor(x))
    t = torch.stack(t_list)
    # (B,H,W,C) -> (B,C,H,W)
    if t.ndim == 4 and t.shape[-1] in (1, 3):
        t = t.permute(0, 3, 1, 2).contiguous()
    return t

def to_bchw1_debug(im, image_processor, device, do_print=False):
    """
    Convert PIL.Image -> (1,C,H,W).
    If do_print=True, print basic sanity info once.
    """
    if do_print:
        print(f"[SANITY] Loaded image size: {im.size}, mode: {im.mode}")

    out = image_processor(im)
    # dict with pixel_values
    if isinstance(out, dict) and "pixel_values" in out:
        px = out["pixel_values"]
        if isinstance(px, torch.Tensor):
            t = px if px.ndim == 4 else px.unsqueeze(0)
        elif isinstance(px, (list, tuple)):
            t = _stack_like(px)
        else:
            raise TypeError(f"Unsupported pixel_values type: {type(px)}")
    # tensor
    elif isinstance(out, torch.Tensor):
        t = out if t.ndim == 4 else out.unsqueeze(0)
    # list/tuple
    elif isinstance(out, (list, tuple)):
        t = _stack_like(out)
    else:
        # Fallback CLIP-like transform
        import torchvision.transforms as T
        tf = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                        std=[0.26862954, 0.26130258, 0.27577711]),
        ])
        t = tf(im).unsqueeze(0)

    if t.ndim == 4 and t.shape[-1] in (1, 3):  # (B,H,W,C) -> (B,C,H,W)
        t = t.permute(0,3,1,2).contiguous()
    if t.ndim == 3:
        t = t.unsqueeze(0)

    if do_print:
        print("[SANITY] img_bchw shape:", t.shape)

    return t.to(device)

def make_vision_x6d(img_bchw, model, device, T_img=1, F=1, do_print=False):
    """Cast to vision encoder dtype and convert to (B,T_img,F,C,H,W) for MedFlamingo."""
    assert img_bchw.ndim == 4, f"expected BCHW, got {img_bchw.ndim}D"
    try:
        vision_dtype = next(model.vision_encoder.parameters()).dtype
    except Exception:
        vision_dtype = img_bchw.dtype
    img_bchw = img_bchw.to(device=device, dtype=vision_dtype)
    v = img_bchw.unsqueeze(1).unsqueeze(2)  # (B,1,1,C,H,W)
    if T_img != 1 or F != 1:
        v = v.repeat(1, T_img, F, 1, 1, 1)

    if do_print:
        print("[SANITY] vision_x shape:", v.shape, "| dtype:", v.dtype)

    return v

# ----- stop criteria + cleaner to prevent truncation mess -----
STOP_STRINGS = ["<|endofchunk|>", "\nQ:", "\nQuestion:", "\n\n"]
_stop_token_ids = [tokenizer(s, add_special_tokens=False).input_ids for s in STOP_STRINGS]
_stop_token_ids = [ids if isinstance(ids, list) else [ids] for ids in _stop_token_ids]

class StopOnTokens(StoppingCriteria):
    def __init__(self, stop_seqs):
        super().__init__()
        self.stop_seqs = stop_seqs  # list[list[int]]

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs):
        for stop in self.stop_seqs:
            L = len(stop)
            if L == 0:
                continue
            if input_ids.shape[1] >= L and input_ids[0, -L:].tolist() == stop:
                return True
        return False

stops = StoppingCriteriaList([StopOnTokens(_stop_token_ids)])

def clean_answer(text: str, question: str = "") -> str:
    """Strip prompt, stop tokens, and question echoes from raw decoded text."""
    # 1) cut at stop strings
    cut = len(text)
    for s in STOP_STRINGS:
        i = text.find(s)
        if i != -1:
            cut = min(cut, i)
    text = text[:cut]

    # 2) take only after last "A:" if present
    if "A:" in text:
        text = text.split("A:", 1)[1]

    text = text.strip().strip(" ,.;")

    # 3) remove verbatim question if it's echoed inside
    if question:
        text = text.replace(question, "").strip()

    # 4) strip leading Q/Question prefixes if any
    for prefix in ["Q:", "Question:", "question:", "Q )"]:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()

    # 5) if empty or too short, standardize as unsure
    if len(text) == 0:
        text = "I am unsure."
    return text

# one-time global flag to print shapes on first prediction
_PRINT_DEBUG_ONCE = True

# -------------------- PREDICT ONE --------------------
def predict_one(image_path, question, model, tokenizer, image_processor, device,
                max_new_tokens=MAX_NEW_TOKENS, max_input_tokens=MAX_INPUT_TOKENS):
    global _PRINT_DEBUG_ONCE

    im = Image.open(image_path).convert("RGB")
    img_bchw = to_bchw1_debug(im, image_processor, device, do_print=_PRINT_DEBUG_ONCE)  # (1,C,H,W)
    vision_x = make_vision_x6d(img_bchw, model, device, do_print=_PRINT_DEBUG_ONCE)    # (1,1,1,C,H,W)

    # Better radiology-aware prompt for MedFlamingo
    prompt = (
        "<image>\n"
        "You are an expert radiologist describing findings on this spine X-ray.\n"
        f"Q: {question}\n"
        "A:"
    )

    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_input_tokens)
    enc = {k: v.to(device) for k, v in enc.items()}

    if _PRINT_DEBUG_ONCE:
        print("[SANITY] lang input_ids shape:", enc["input_ids"].shape)
        print("[SANITY] prompt tokens:", enc["input_ids"].shape[1])
        _PRINT_DEBUG_ONCE = False

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        num_beams=1,
        do_sample=False,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY,
        use_cache=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        stopping_criteria=stops,
    )

    with torch.no_grad():
        try:
            amp_dtype = next(model.parameters()).dtype
        except StopIteration:
            amp_dtype = torch.float16
        with torch.cuda.amp.autocast(enabled=(device == "cuda"), dtype=amp_dtype):
            out = model.generate(
                vision_x=vision_x,
                lang_x=enc["input_ids"],
                attention_mask=enc.get("attention_mask", None),
                **gen_kwargs,
            )

    full = tokenizer.decode(out[0], skip_special_tokens=False)

    # optional: cut at first <|endofchunk|> if model emits it
    if "<|endofchunk|>" in full:
        full = full.split("<|endofchunk|>", 1)[0]

    ans = clean_answer(full, question=question)

    # Clean up
    del vision_x, img_bchw, enc, out
    if device == "cuda":
        torch.cuda.empty_cache()
    return ans

# -------------------- RUN ON TEST SET --------------------
rows = []

print(">> Starting zero-shot inference on test set...")
with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(tqdm(f, desc="Zero-shot (test)")):
        rec = json.loads(line)
        image_path = rec["image_path"]
        question   = rec["question"]
        gt         = rec.get("answer", "")

        try:
            pred = predict_one(
                image_path=image_path,
                question=question,
                model=model,
                tokenizer=tokenizer,
                image_processor=image_processor,
                device=device,
                max_new_tokens=MAX_NEW_TOKENS,
                max_input_tokens=MAX_INPUT_TOKENS,
            )
        except Exception as e:
            print(f"[ERROR] {image_path}: {e}")
            pred = f"ERROR: {e}"

        # small sanity print for first few rows
        if i < 3:
            print("\n[SANITY-SAMPLE]", i)
            print("  image_path:", image_path)
            print("  Q:", question)
            print("  GT:", gt)
            print("  P :", pred)

        rows.append({
            "image_path": image_path,
            "question": question,
            "ground_truth": gt,
            "prediction": pred,
        })

        if device == "cuda":
            torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print("✅ Saved to", OUT_CSV.resolve())


Device: cuda


/home/deepalim/work/MedFlamingo/Medflaming_env/lib/python3.10/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior wil

Flamingo model initialized with 4657367104 trainable parameters
>> Loading checkpoint from: checkpoints/openflamingo-9b/checkpoint.pt


/tmp/ipykernel_1271283/365228492.py:60: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(CHECKPOINT_PATH, map_location="cpu")


OutOfMemoryError: CUDA out of memory. Tried to allocate 172.00 MiB. GPU 0 has a total capacity of 47.40 GiB of which 135.94 MiB is free. Process 3271070 has 2.30 GiB memory in use. Process 269377 has 3.60 GiB memory in use. Including non-PyTorch memory, this process has 41.34 GiB memory in use. Of the allocated memory 41.01 GiB is allocated by PyTorch, and 21.41 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [2]:
# %% Install (uncomment & run once in your env)
# %pip install pandas nltk rouge-score bert-score

import re
from pathlib import Path

import pandas as pd
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score


# =========================
# CONFIG
# =========================
CSV_PATH = Path("./ZeroShot_test.csv")

REF_COL = "ground_truth"             # ground-truth column name
PRED_COL = "prediction"  # prediction column name

# If some rows are clearly invalid predictions, you can filter by a prefix:
ERROR_PREFIX = "[ERROR]"       # set to None if you don't want this filter

# BERTScore config
BERT_MODEL_TYPE = "bert-base-uncased"  # you can switch to a larger model later
#BERT_LANG = "en"
BERT_BATCH_SIZE = 64                  # adjust based on your GPU memory
#BERT_RESCALE_BASELINE = True          # recommended for English


# =========================
# HELPERS
# =========================
def normalize_text(s: str) -> str:
    """Lowercase + strip + collapse spaces. Extend if you want more aggressive cleaning."""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def load_and_prepare(csv_path: Path, ref_col: str, pred_col: str, error_prefix: str | None = None):
    df = pd.read_csv(csv_path)

    # Drop rows with missing ref or pred
    df = df.dropna(subset=[ref_col, pred_col])

    # Optional: drop obvious error rows (e.g. "[ERROR] rate limited")
    if error_prefix:
        df = df[~df[pred_col].astype(str).str.startswith(error_prefix)]

    df = df.reset_index(drop=True)

    refs_raw = df[ref_col].astype(str).tolist()
    preds_raw = df[pred_col].astype(str).tolist()

    print(f"Using {len(refs_raw)} examples for evaluation")
    return refs_raw, preds_raw


def compute_accuracy(ref_norm, pred_norm):
    correct = sum(r == p for r, p in zip(ref_norm, pred_norm))
    total = len(ref_norm)
    acc = correct / total if total > 0 else 0.0
    return acc, correct, total


def compute_bleu(ref_norm, pred_norm):
    # NLTK expects list of list of tokenized references, and list of tokenized hypotheses
    refs_tok = [[r.split()] for r in ref_norm]   # one reference per example
    preds_tok = [p.split() for p in pred_norm]

    smooth = SmoothingFunction().method1
    bleu4 = corpus_bleu(
        refs_tok,
        preds_tok,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smooth,
    )
    return bleu4


def compute_rouge_l(ref_norm, pred_norm):
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    f_scores = []

    for r, p in zip(ref_norm, pred_norm):
        score = scorer.score(r, p)["rougeL"]
        f_scores.append(score.fmeasure)

    avg_f = sum(f_scores) / len(f_scores) if f_scores else 0.0
    return avg_f


def compute_bertscore(refs_raw, preds_raw):
    print("\nComputing BERTScore (this may take a bit)...")
    P, R, F1 = bertscore_score(
        cands=preds_raw,
        refs=refs_raw,
        model_type=BERT_MODEL_TYPE,
        #lang=BERT_LANG,
        batch_size=BERT_BATCH_SIZE,
        #rescale_with_baseline=BERT_RESCALE_BASELINE,
        verbose=True,
    )

    bert_p = float(P.mean())
    bert_r = float(R.mean())
    bert_f1 = float(F1.mean())
    return bert_p, bert_r, bert_f1


# =========================
# MAIN
# =========================
if __name__ == "__main__":
    # 1. Load data
    refs_raw, preds_raw = load_and_prepare(CSV_PATH, REF_COL, PRED_COL, ERROR_PREFIX)

    # 2. Normalize (for Accuracy, BLEU, ROUGE)
    ref_norm = [normalize_text(x) for x in refs_raw]
    pred_norm = [normalize_text(x) for x in preds_raw]

    # 3. Accuracy
    accuracy, correct, total = compute_accuracy(ref_norm, pred_norm)
    print(f"\nAccuracy (exact match, normalized): {accuracy:.4f}  ({correct}/{total})")

    # 4. BLEU-4
    bleu4 = compute_bleu(ref_norm, pred_norm)
    print(f"BLEU-4 (corpus, normalized): {bleu4:.4f}")

    # 5. ROUGE-L (F1)
    rouge_l_f1 = compute_rouge_l(ref_norm, pred_norm)
    print(f"ROUGE-L (F1, avg, normalized): {rouge_l_f1:.4f}")

    # 6. BERTScore (raw text)
    bert_p, bert_r, bert_f1 = compute_bertscore(refs_raw, preds_raw)
    print(f"BERTScore Precision (mean): {bert_p:.4f}")
    print(f"BERTScore Recall    (mean): {bert_r:.4f}")
    print(f"BERTScore F1        (mean): {bert_f1:.4f}")

    # 7. Summary dict (easy to log / save)
    metrics = {
        "accuracy_exact": accuracy,
        "bleu4": bleu4,
        "rougeL_f1": rouge_l_f1,
        "bertscore_p": bert_p,
        "bertscore_r": bert_r,
        "bertscore_f1": bert_f1,
    }

    print("\n=== Summary Metrics ===")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")


Using 535 examples for evaluation

Accuracy (exact match, normalized): 0.0000  (0/535)
BLEU-4 (corpus, normalized): 0.0079
ROUGE-L (F1, avg, normalized): 0.0750

Computing BERTScore (this may take a bit)...
calculating scores...
computing bert embedding.


  0%|          | 0/12 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/9 [00:00<?, ?it/s]

done in 7.18 seconds, 74.46 sentences/sec
BERTScore Precision (mean): 0.3858
BERTScore Recall    (mean): 0.4528
BERTScore F1        (mean): 0.4129

=== Summary Metrics ===
accuracy_exact: 0.0000
bleu4: 0.0079
rougeL_f1: 0.0750
bertscore_p: 0.3858
bertscore_r: 0.4528
bertscore_f1: 0.4129
